# Next Word Prediction with GPT2

In [1]:
MODEL_ID = "openai-community/gpt2"

## Tokenization

In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

########## Let have a look at the vocabulary of GPT-2 ##########
vocab = tokenizer.get_vocab() #key=subword, value=token_id
print(f"vocabulary is of type {type(vocab)} with size {len(vocab)}")

vocabulary is of type <class 'dict'> with size 50257


In [3]:
prompt = "The theory of relativity is"

In [4]:
input_ids = tokenizer.encode(prompt)
input_ids

[464, 4583, 286, 44449, 318]

In [5]:
for token_id in input_ids:
    subword = tokenizer.decode(token_id)
    print(token_id, "\t->", f"'{subword}'")

464 	-> 'The'
4583 	-> ' theory'
286 	-> ' of'
44449 	-> ' relativity'
318 	-> ' is'


## Plain GPT2 
![without_language_model_head](./without_language_model_head.png)

In [6]:
from transformers import AutoModel
model = AutoModel.from_pretrained(MODEL_ID)

In [7]:
print(model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [8]:
# Compute The Hidden States (Last vector embedding before applying the language head)
import torch
input_ids = tokenizer.encode(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = model(input_ids=input_ids, output_attentions=False)
last_hidden_state = outputs.last_hidden_state # returns the logits of last hidden state

In [9]:
last_hidden_state.shape # batch, sequence, embedding dimension

torch.Size([1, 5, 768])

## Understanding Text Generation
For "next word prediction" put a language model head on top of GPT2: Use GPT2LMHeadModel, since this adds the language modeling head on top of GPT2Model.
![with_language_model_head](with_language_model_head.png)

In [10]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [11]:
prompt = "The theory of relativity is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# logits is a tensor of shape (batch_size, sequence_length, vocab_size)
logits = outputs.logits
print(logits.shape)

torch.Size([1, 5, 50257])


### Use Softmax to convert logits into probabilities

Apply torch.softmax(logits, dim=-1) and get probabilities.

https://de.wikipedia.org/wiki/Softmax-Funktion

In [12]:
probabilities = torch.softmax(logits[0, :, :], dim=-1)
probabilities.shape

torch.Size([5, 50257])

In [13]:
#Use the probability from the last time step!
next_token_probs = probabilities[-1,:]
print(next_token_probs.shape) # for each token in the vocabulary we get a probability

torch.Size([50257])


In [14]:
print("get the top 10 and put all together")
topk_next_tokens= torch.topk(next_token_probs, 10) 
topk_next_token_list = [(tokenizer.decode(idx), prob) for idx, prob in zip(topk_next_tokens.indices, topk_next_tokens.values)] 
for token, prob in topk_next_token_list:
    print(round(prob.item(),3), token)

get the top 10 and put all together
0.381  that
0.081  based
0.04  the
0.038  a
0.028  not
0.019  simple
0.013  to
0.01 ,
0.01  very
0.009  one


In [15]:
index = torch.argmax(next_token_probs)
next_token = index.item()
print("index", next_token, "with probability", next_token_probs[next_token].item()) #index == token_id
tokenizer.decode(next_token)

index 326 with probability 0.3814721405506134


' that'

## Greedy Search

In [18]:
vocab.items()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


dict_items([('oshi', 13704), ('Ġreverence', 40380), ('Ġserpent', 33727), ('Ġposit', 46436), ('padding', 39231), ('ĠKatrina', 33251), ('Ġphony', 41421), ('ĠRos', 10018), ('ĠZap', 36079), ('Ġmotors', 24699), ('ructose', 32275), ('Mal', 15029), ('ĠInterview', 19371), ('pos', 1930), ('LECT', 16779), ('Prin', 47231), ('Ġvisible', 7424), ('ĠPAGE', 48488), ('blogs', 49096), ('Ġconfig', 4566), ('Ġace', 31506), ('about', 10755), ('RAL', 35296), ('ĠHungry', 42939), ('ĠJosÃ©', 36997), ('Ġskept', 11200), ('ĠNeeds', 36557), ('iqueness', 46764), ('ĠDiscipline', 48532), ('Ġcircumst', 5397), ('Ġdissolution', 37494), ('Ġshooting', 4395), ('Ġembodiment', 23168), ('Ġrodents', 41093), ('Ġgoverns', 47049), ('inge', 11912), ('doc', 15390), ('Ġinspir', 32285), ('474', 38652), ('Ġexclaim', 48775), ('Ġdisorders', 11916), ('ĠRae', 41155), ('ĠShelby', 35053), ('ĠCells', 39794), ('Ġripped', 19551), ('588', 39118), ('JD', 37882), ('Ġdisc', 1221), ('Ġfairy', 25607), ('Ġsuggestion', 13052), ('NOT', 11929), ('rosse',

In [16]:
############# greedy search loop ###########
prompt = "The theory of relativity is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
max_new_tokens = 15

for i in range(max_new_tokens):
    outputs = model(input_ids=input_ids)
    next_token_logits = outputs.logits[0, -1,:]
    next_token_probs = torch.softmax(next_token_logits, -1)
    index = torch.argmax(next_token_probs)
    prompt = prompt + tokenizer.decode(index.item())
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    print(prompt)

The theory of relativity is that
The theory of relativity is that the
The theory of relativity is that the speed
The theory of relativity is that the speed of
The theory of relativity is that the speed of light
The theory of relativity is that the speed of light is
The theory of relativity is that the speed of light is proportional
The theory of relativity is that the speed of light is proportional to
The theory of relativity is that the speed of light is proportional to the
The theory of relativity is that the speed of light is proportional to the distance
The theory of relativity is that the speed of light is proportional to the distance between
The theory of relativity is that the speed of light is proportional to the distance between two
The theory of relativity is that the speed of light is proportional to the distance between two points
The theory of relativity is that the speed of light is proportional to the distance between two points.
The theory of relativity is that the spee

## Beam Search

In [19]:
############# a very naive implementation of beam search with num_beams=2 loop ###########
prompt = "The theory of relativity is"
print(prompt)

max_probs = 5

pairs = []

tokens_as_list = sorted([token_id for _, token_id in vocab.items()])

def generate(inputs):
    with torch.no_grad():
        outputs = model(**inputs)
    next_token_logits = outputs.logits[0, -1,:]
    
    next_token_probs = torch.softmax(next_token_logits, -1)
    probs = [prob.item() for prob in list(next_token_probs)]
    return probs
    
iteration_step = 0
# first iteration
iteration_step += 1
print("iteration_step", iteration_step, "----------")
inputs = tokenizer(prompt, return_tensors="pt")

prob_tokens = sorted(zip(generate(inputs),  tokens_as_list), key=lambda x:x[0], reverse=True)
new_prob_tokens = prob_tokens[0:max_probs]
for new_prob, new_token in new_prob_tokens:
    print(new_prob, new_token, tokenizer.decode(new_token))

The theory of relativity is
iteration_step 1 ----------
0.3814721405506134 326  that
0.08116311579942703 1912  based
0.03981886804103851 262  the
0.03792071342468262 257  a
0.028438160195946693 407  not


In [18]:
############# a very naive implementation of beam search with num_beams=2 loop ###########
prompt = "The theory of relativity is"
print(prompt)

max_probs = 5

pairs = []

tokens_as_list = [token_id for _, token_id in vocab.items()]

def generate(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    next_token_logits = outputs.logits[0, -1,:]
    next_token_probs = torch.softmax(next_token_logits, -1)
    probs = [prob.item() for prob in list(next_token_probs)]
    return probs

iteration_step = 0
# first iteration
iteration_step += 1
print("iteration_step", iteration_step, "----------")
input_ids = tokenizer.encode(prompt, return_tensors="pt")

prob_tokens = sorted(zip(generate(input_ids),  tokens_as_list), key=lambda x:x[0], reverse=True)
new_prob_tokens = prob_tokens[0:max_probs]
for new_prob, new_token in new_prob_tokens:
    print(new_prob, new_token, tokenizer.decode(new_token))

# second iteration
prob_tokens = prob_tokens[0:max_probs]
iteration_step += 1
pairs = []
print("iteration_step", iteration_step, "----------")
for prob, token in prob_tokens:
    new_prompt = prompt + tokenizer.decode(token)
    input_ids = tokenizer.encode(new_prompt, return_tensors="pt")
    new_prob_tokens = sorted(zip(generate(input_ids),  tokens_as_list), key=lambda x:x[0], reverse=True)
    #print("  ", new_prob_tokens[0:max_probs])
    new_prob_tokens = new_prob_tokens[0:max_probs]
    for new_prob, new_token in new_prob_tokens:
        pairs.append((prob*new_prob, (prob, new_prob), [token, new_token]))

for (prob, (prob1, prob2), tokens) in sorted(pairs,key=lambda x:x[0], reverse=True):
    print(prob, "=(", prob1, "*", prob2, ")", tokens, tokenizer.decode(tokens))
    
    
    
    

The theory of relativity is
iteration_step 1 ----------
0.3814721405506134 48337 Jamie
0.08116311579942703 31076  fetus
0.03981886804103851 26129 ategories
0.03792071342468262 13995  pledge
0.028438160195946693 14670 ORK
iteration_step 2 ----------
0.041581452165743826 =( 0.08116311579942703 * 0.5123195648193359 ) [31076, 4597]  fetusady
0.011531328210785341 =( 0.3814721405506134 * 0.030228493735194206 ) [48337, 22510] Jamienum
0.0070805449036347445 =( 0.3814721405506134 * 0.01856110617518425 ) [48337, 40541] JamieMaximum
0.005815147453009928 =( 0.03792071342468262 * 0.15335015952587128 ) [13995, 28625]  pledge heir
0.004498780062135999 =( 0.3814721405506134 * 0.011793207377195358 ) [48337, 24119] JamieMary
0.004416985772779131 =( 0.3814721405506134 * 0.011578789912164211 ) [48337, 21854] Jamie prevalent
0.004257903160327037 =( 0.03981886804103851 * 0.10693179816007614 ) [26129, 3523] ategories citiz
0.0038899529661726717 =( 0.03792071342468262 * 0.1025812178850174 ) [13995, 4597]  ple

## Generate Method

In [20]:
############# Generate Text ###########
# the generate method is dooing the looping
prompt = "The theory of relativity is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
attention_mask = torch.ones_like(input_ids)
outputs = model.generate(input_ids, attention_mask=attention_mask,
                         pad_token_id = tokenizer.eos_token_id,
                         max_new_tokens=100, num_beams=3)
tokenizer.decode(outputs[0])

'The theory of relativity is that the speed of light is proportional to the speed of light, and that the speed of light is proportional to the speed of light.\n\nThe theory of relativity is that the speed of light is proportional to the speed of light, and that the speed of light is proportional to the speed of light.\n\nThe theory of relativity is that the speed of light is proportional to the speed of light, and that the speed of light is proportional to the speed of light.\n\nThe theory of'

In [21]:
# the generate method with beam serach und temperature
prompt = "The theory of relativity is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
attention_mask = torch.ones_like(input_ids)
outputs = model.generate(input_ids, attention_mask=attention_mask,
                         pad_token_id = tokenizer.eos_token_id,
                         max_new_tokens=100, num_beams=4, 
                         temperature=2.0, do_sample=True,
                         repetition_penalty=10.0
                        )
tokenizer.decode(outputs[0])

"The theory of relativity is based on the observation that, for every second in an object's velocity (or angular momentum) you accelerate to a certain speed at which it moves through space. If this accelerates faster or slower than your time constant then one could have trouble determining exactly how fast something moved and whether its motion would be affected by any change in velocities such as gravitational acceleration. And if things were moving so quickly when they had reached those speeds their motions wouldn't cause significant amounts of perturbation due to some"

## Perplexity

Perplexity (PPL) is computed as: 

$$PPL = e^{loss}$$
 
where loss is the cross-entropy loss returned by the model.

In [22]:
import math
s_1 = "The theory of relativity is cool."
s_2 = "The theory of relativity is great."
sentences = [s_1, s_2]

for sentence in sentences:
    print(sentence)
    inputs = tokenizer(sentence, return_tensors="pt")

    # GPT-2 needs labels to compute loss
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss

    # Compute perplexity
    perplexity = math.exp(loss.item())
    print(f"Loss      : {loss.item():.4f}")
    print(f"Perplexity: {perplexity:.4f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


The theory of relativity is cool.
Loss      : 4.9085
Perplexity: 135.4368
The theory of relativity is great.
Loss      : 4.6448
Perplexity: 104.0434


# Backup

In [ ]:
# How does GPT decide to stop generating sentences without EOS token?
# see https://stackoverflow.com/questions/77549942/stopping-criteria-for-llama-2-does-not-work